In [ ]:
from google.colab import files
import pandas as pd #Biblioteca para ciência de dados
import numpy as np
import io

print("Selecione o arquivo CSV que deseja analisar:")
uploaded = files.upload()

nome_arquivo = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

print(f"\nArquivo '{nome_arquivo}' carregado com sucesso!")
print(f"Dimensoes: {df.shape[0]} linhas x {df.shape[1]} colunas\n")

def checar_espacos_em_branco(df):
    """Verifica espacos em branco extras em colunas de texto."""
    problemas = {}
    for col in df.select_dtypes(include='object').columns: #checa os espaços em branco do texto (object)
        serie = df[col].dropna().astype(str) #dropna() remove os vazios e garante que seja tratado como teto astype(str)
        com_espaco_borda = serie[serie != serie.str.strip()] #.str.strip() remove espaço vazio do início e fim
        com_espaco_duplo = serie[serie.str.contains(r'\s{2,}', regex=True, na=False)] #regex procura palavras com 2 ou mais espaços
        total = len(com_espaco_borda) + len(com_espaco_duplo)
        if total > 0:
            problemas[col] = {
                'espacos_nas_bordas': len(com_espaco_borda),
                'espacos_duplos_no_meio': len(com_espaco_duplo)
            }
    return problemas


def checar_padronizacao(df, limite_categorias=30):
    """
    Verifica se colunas de texto tem valores que parecem ser
    a mesma coisa escrita de forma diferente (ex: 'sp', 'SP', 'Sao Paulo').
    So analisa colunas com poucas categorias unicas (evita IDs e textos livres).
    """
    problemas = {}
    for col in df.select_dtypes(include='object').columns:
        valores_unicos = df[col].dropna().astype(str).unique()
        if len(valores_unicos) == 0 or len(valores_unicos) > limite_categorias:
            continue

        normalizados = {}
        for v in valores_unicos:
            chave = v.strip().lower() #tira espaços e deixa tudo em minúsculo
            normalizados.setdefault(chave, []).append(v) #agrupa tudo na mesma chave

        grupos_inconsistentes = {k: v for k, v in normalizados.items() if len(v) > 1}
        if grupos_inconsistentes:
            problemas[col] = grupos_inconsistentes
    return problemas


def checar_valores_ausentes(df):
    """Verifica valores nulos e 'pseudo-nulos' como 'N/A', '-', etc."""
    pseudo_nulos = ['n/a', 'na', 'nan', 'null', '-', '--', 'none', '', ' '] #checa todas as variações de valores nulos
    resultado = {}
    for col in df.columns:
        nulos_reais = df[col].isna().sum() #quantas são nulas de verdade
        if df[col].dtype == 'object':
            pseudo = df[col].dropna().astype(str).str.strip().str.lower().isin(pseudo_nulos).sum()
        else:
            pseudo = 0
        total = nulos_reais + pseudo
        if total > 0:
            resultado[col] = {
                'nulos_reais': int(nulos_reais),
                'pseudo_nulos': int(pseudo),
                'percentual': round(total / len(df) * 100, 1) #porcentagem de nulos ausentes na coluna para analisar se vale a pena descartar
            }
    return resultado


def checar_duplicatas(df):
    """Verifica linhas completamente duplicadas."""
    duplicadas = df[df.duplicated(keep=False)] #df.duplicated() marca as duplicatas. keep=false conta todas as duplicadas, não apenas depois da segunda
    return len(duplicadas), duplicadas


def checar_tipos_inconsistentes(df):
    """
    Verifica colunas de texto que parecem conter numeros
    misturados com texto (ex: '100', 'cem', '1.000').
    """
    problemas = {}
    for col in df.select_dtypes(include='object').columns:
        serie = df[col].dropna().astype(str)
        parece_numero = serie.str.match(r'^-?\d+([.,]\d+)?$') #regex verifica se o texto parece número
        if 0 < parece_numero.sum() < len(serie):
            problemas[col] = {
                'parecem_numero': int(parece_numero.sum()),
                'nao_numero': int((~parece_numero).sum())
            }
    return problemas


# --- PASSO 3: Executar analises ---
relatorio = []
relatorio.append("=" * 60)
relatorio.append("RELATORIO DE QUALIDADE DE DADOS")
relatorio.append(f"Arquivo: {nome_arquivo}")
relatorio.append(f"Dimensoes: {df.shape[0]} linhas x {df.shape[1]} colunas")
relatorio.append("=" * 60)

relatorio.append("\n1. ESPACOS EM BRANCO EXTRAS")
espacos = checar_espacos_em_branco(df)
if espacos:
    for col, info in espacos.items():
        relatorio.append(f"  - Coluna '{col}': {info['espacos_nas_bordas']} valores com espaco "
                          f"no inicio/fim, {info['espacos_duplos_no_meio']} com espaco duplo no meio")
else:
    relatorio.append("  Nenhum problema encontrado.")

relatorio.append("\n2. FALTA DE PADRONIZACAO (valores que podem ser a mesma coisa)")
padronizacao = checar_padronizacao(df)
if padronizacao:
    for col, grupos in padronizacao.items():
        relatorio.append(f"  - Coluna '{col}':")
        for chave, variacoes in grupos.items():
            relatorio.append(f"      * Possivel mesmo valor escrito como: {variacoes}")
else:
    relatorio.append("  Nenhum problema encontrado.")

relatorio.append("\n3. VALORES AUSENTES")
ausentes = checar_valores_ausentes(df)
if ausentes:
    for col, info in ausentes.items():
        relatorio.append(f"  - Coluna '{col}': {info['nulos_reais']} nulos reais, "
                          f"{info['pseudo_nulos']} pseudo-nulos (ex: 'N/A', '-') "
                          f"- {info['percentual']}% da coluna")
else:
    relatorio.append("  Nenhum problema encontrado.")

relatorio.append("\n4. LINHAS DUPLICADAS")
qtd_dup, linhas_dup = checar_duplicatas(df)
if qtd_dup > 0:
    relatorio.append(f"  - {qtd_dup} linhas duplicadas encontradas (indices: {list(linhas_dup.index)})")
else:
    relatorio.append("  Nenhuma duplicata encontrada.")

relatorio.append("\n5. POSSIVEIS TIPOS DE DADOS INCONSISTENTES")
tipos = checar_tipos_inconsistentes(df)
if tipos:
    for col, info in tipos.items():
        relatorio.append(f"  - Coluna '{col}': {info['parecem_numero']} valores parecem numero, "
                          f"{info['nao_numero']} parecem texto - pode ser coluna mista")
else:
    relatorio.append("  Nenhum problema encontrado.")

relatorio.append("\n" + "=" * 60)

texto_relatorio = "\n".join(relatorio)
print(texto_relatorio)

with open("relatorio_qualidade.txt", "w", encoding="utf-8") as f:
    f.write(texto_relatorio)

print("\nRelatorio salvo como 'relatorio_qualidade.txt'")


# --- PASSO 4: Gerar versao corrigida (apenas correcoes seguras) ---
df_corrigido = df.copy()

for col in df_corrigido.select_dtypes(include='object').columns:
    df_corrigido[col] = df_corrigido[col].astype(str).str.strip()
    df_corrigido[col] = df_corrigido[col].str.replace(r'\s{2,}', ' ', regex=True)
    df_corrigido[col] = df_corrigido[col].replace('nan', np.nan)

df_corrigido.to_csv("arquivo_corrigido.csv", index=False)

Selecione o arquivo CSV que deseja analisar:


Saving transformed_datatran2025_5.csv to transformed_datatran2025_5.csv
Saving transformed_datatran2026_5.csv to transformed_datatran2026_5.csv

Arquivo 'transformed_datatran2025_5.csv' carregado com sucesso!
Dimensoes: 72529 linhas x 25 colunas

RELATORIO DE QUALIDADE DE DADOS
Arquivo: transformed_datatran2025_5.csv
Dimensoes: 72529 linhas x 25 colunas

1. ESPACOS EM BRANCO EXTRAS
  - Coluna 'municipio': 0 valores com espaco no inicio/fim, 12 com espaco duplo no meio

2. FALTA DE PADRONIZACAO (valores que podem ser a mesma coisa)
  Nenhum problema encontrado.

3. VALORES AUSENTES
  - Coluna 'classificacao_acidente': 1 nulos reais, 0 pseudo-nulos (ex: 'N/A', '-') - 0.0% da coluna
  - Coluna 'delegacia': 22 nulos reais, 0 pseudo-nulos (ex: 'N/A', '-') - 0.0% da coluna
  - Coluna 'regional': 2 nulos reais, 0 pseudo-nulos (ex: 'N/A', '-') - 0.0% da coluna
  - Coluna 'uop': 38 nulos reais, 0 pseudo-nulos (ex: 'N/A', '-') - 0.1% da coluna

4. LINHAS DUPLICADAS
  Nenhuma duplicata encontrada